In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import KNNImputer
from sklearn.metrics import mean_absolute_error
from sklearn.decomposition import PCA
import random
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVR
from xgboost import XGBRegressor


SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PATH = '/kaggle/input/competitions/home-data-for-ml-course'

df = pd.read_csv(f'{PATH}/train.csv')
test_df = pd.read_csv(f'{PATH}/test.csv')
ids = test_df.Id
test_df = test_df.drop('Id', axis=1, inplace=False)

In [ ]:
y = df['SalePrice']
x = df.drop(['Id', 'SalePrice'], axis=1, inplace=False)
not_n = x.select_dtypes(include=['object'])
for each in not_n: 
    x[each] = x[each].astype('category').cat.codes
    test_df[each] = test_df[each].astype('category').cat.codes

x_train, x_val, y_train, y_val = train_test_split(x, y, train_size=0.8)

In [ ]:
svr_pipeline = make_pipeline(
    KNNImputer(n_neighbors=5),
    StandardScaler(),
    SVR(kernel='rbf')
)

In [ ]:
forest_pipeline = make_pipeline(
    KNNImputer(n_neighbors=5),
    RandomForestRegressor(random_state=SEED, n_estimators=100)
)

In [ ]:
xgb_pipeline = make_pipeline(
    KNNImputer(n_neighbors=5),
    StandardScaler(),
    XGBRegressor(n_estimators=100, random_state=SEED)
)

In [ ]:
def validate(pipe):
    pipe.fit(x_train, y_train)
    print(mean_absolute_error(y_val, pipe.predict(x_val)))

In [ ]:
validate(svr_pipeline)
validate(forest_pipeline)
validate(xgb_pipeline)

In [ ]:
best_pipe = xgb_pipeline.fit(x, y)
pred = best_pipe.predict(test_df)

In [ ]:
submission = pd.DataFrame({
    'Id' : ids,
    'SalePrice': pred
})

submission.to_csv('pred.csv', index=False)